# Step 3: Multi-Scholar Semantic Search & AI Synthesis

This notebook provides a complete, runnable end-to-end pipeline for the upgraded **Multi-Scholar Islamic RAG System**.

### Key Capabilities:
1. **Multi-Scholar Dataset Support**: Indexes **40,062 fatwas** across major Sunni/Salafi scholars and official bodies:
   - **Sheikh Abdulaziz ibn Baz**
   - **Sheikh Muhammad ibn Saalih al-Uthaymeen**
   - **Sheikh Saleh Al-Fawzan**
   - **Al-Lajnah Ad-Da'imah lil-Ifta** (Permanent Committee / Council of Senior Scholars)
   - **Sheikh Abdulaziz Aal ash-Sheikh** (Grand Mufti)
   - **Sheikh Abdullah al-Manee'**
   - **Sheikh Saleh bin Humaid**
2. **Two Search Modes**:
   - **Mode 1**: **X answers per scholar** (searches and reranks each scholar's corpus independently to guarantee balanced representation).
   - **Mode 2**: **X answers ranked overall** (searches and reranks across all selected scholars as a unified pool).
3. **Two-Stage Neural Retrieval**: Precomputed `Qwen3-Embedding-0.6B` vectors + `ALJIACHI/Mizan-Rerank-V2` cross-encoder.
4. **AI Review & Judging**: Per-fatwa grounded verdict (`answers-it`, `related-only`, `unrelated`).
5. **New AI Synthesis Features**:
   - **Consensus Classification** (`agree` / `partially-agree` / `disagree`) evaluated using **Gemini** or **JEV** (`typesafe/jev-*`).
   - **Comparative Summary**: A coherent synthesis of what the scholars say and rule, generated via **Gemini**.

In [1]:
# 1. Setup & Environment
!pip install -q sentence-transformers transformers google-genai torch pandas ftfy

import os, sys, json, time, re
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from sentence_transformers import SentenceTransformer, CrossEncoder

# Load environment variables from .env if present
for env_path in [Path('.env'), Path('../.env')]:
    if env_path.exists():
        with open(env_path, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if line and not line.startswith('#') and '=' in line:
                    k, v = line.split('=', 1)
                    os.environ.setdefault(k.strip(), v.strip().strip('"\''))

gemini_ok = bool(os.getenv('GEMINI_API_KEY'))
openrouter_ok = bool(os.getenv('OPENROUTER_API_KEY'))
print(f'Gemini API Key:     {"✓ Set" if gemini_ok else "✕ Missing (set GEMINI_API_KEY)"}')
print(f'OpenRouter API Key: {"✓ Set" if openrouter_ok else "✕ Missing (set OPENROUTER_API_KEY)"}')
print(f'CUDA available:     {torch.cuda.is_available()}')

fish: Unknown command: pip
fish: 
pip install -q sentence-transformers transformers google-genai torch pandas ftfy
^~^


Gemini API Key:     ✓ Set
OpenRouter API Key: ✓ Set
CUDA available:     True


In [2]:
# 2. Load Multi-Scholar Assets (with Graceful Fallback)
DATA_DIRS = [Path('data'), Path('../data'), Path('data/cleaned'), Path('../data/cleaned')]

def find_file(filename: str) -> Path | None:
    for d in DATA_DIRS:
        p = d / filename
        if p.exists():
            return p
    return None

# Look for multi-scholar artifacts, or fall back to Bin Baz if not found
multi_csv = find_file('multi_scholar_clean.csv')
multi_emb = find_file('qwen3_06b_multi_scholar_embeddings.pt')
binbaz_csv = find_file('binbaz_clean.csv')
binbaz_emb = find_file('qwen3_06b_title_answer_embeddings.pt')

if multi_csv and multi_emb:
    print(f'Loading Multi-Scholar Corpus: {multi_csv}')
    df = pd.read_csv(multi_csv)
    embeddings = torch.load(multi_emb, map_location='cpu', weights_only=True)
elif binbaz_csv and binbaz_emb:
    print('Multi-scholar embeddings not detected — loading existing Bin Baz corpus as base.')
    df = pd.read_csv(binbaz_csv)
    embeddings = torch.load(binbaz_emb, map_location='cpu', weights_only=True)
else:
    raise FileNotFoundError('No valid CSV and embeddings pair found in data directories!')

if isinstance(embeddings, dict):
    for k in ('embeddings', 'tensor', 'data'):
        if k in embeddings:
            embeddings = embeddings[k]
            break
embeddings = F.normalize(torch.as_tensor(embeddings, dtype=torch.float32), p=2, dim=1)
df = df.iloc[:len(embeddings)].copy().reset_index(drop=True)
print(f'Indexed {len(df):,} fatwas · Embeddings matrix shape: {tuple(embeddings.shape)}')
print('\nScholars available in index:')
print(df['source'].value_counts())

Loading Multi-Scholar Corpus: ../data/multi_scholar_clean.csv


Indexed 40,062 fatwas · Embeddings matrix shape: (40062, 1024)

Scholars available in index:
source
Ibn Baz                     19727
Ibn Uthaymeen                9119
Al-Lajnah Ad-Daimah          9086
Saleh Al-Fawzan              1714
Abdullah al-Manee             233
Abdulaziz Aal ash-Sheikh      135
Saleh bin Humaid               48
Name: count, dtype: int64


In [3]:
# 3. Load Retrieval & Reranker Models
# On smaller GPUs (e.g. 3GB GTX 1050), run embedder on CPU (only encodes query, lightweight)
# and reranker on GPU. On larger GPUs (Colab T4 15GB), both can run on CUDA.
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3) if torch.cuda.is_available() else 0
device_embedder = 'cuda' if vram_gb > 6.0 else 'cpu'
device_reranker = 'cuda' if torch.cuda.is_available() else 'cpu'

EMBEDDER_MODEL = 'Qwen/Qwen3-Embedding-0.6B'
RERANKER_MODEL = 'ALJIACHI/Mizan-Rerank-V2'

print(f'Loading embedder ({EMBEDDER_MODEL}) on {device_embedder}...')
embedder = SentenceTransformer(EMBEDDER_MODEL, device=device_embedder)
embedder.max_seq_length = 1024

print(f'Loading reranker ({RERANKER_MODEL}) on {device_reranker}...')
reranker = CrossEncoder(RERANKER_MODEL, max_length=1024, device=device_reranker, trust_remote_code=True)
print('Models ready.')

Loading embedder (Qwen/Qwen3-Embedding-0.6B) on cpu...


Loading reranker (ALJIACHI/Mizan-Rerank-V2) on cuda...


Models ready.


In [4]:
# 4. Search Engines: Mode 1 and Mode 2

def encode_query(query: str) -> torch.Tensor:
    vec = embedder.encode(query, convert_to_tensor=True, device=device_embedder, normalize_embeddings=True)
    return vec.detach().cpu().float()

def rerank(query: str, candidates: pd.DataFrame) -> pd.DataFrame:
    if len(candidates) == 0:
        return candidates
    pairs = [[query, str(t)] for t in candidates['embed_text'].tolist()]
    scores = np.asarray(reranker.predict(pairs, batch_size=16, show_progress_bar=False)).reshape(-1)
    res = candidates.copy()
    res['reranker_score'] = scores
    res = res.sort_values('reranker_score', ascending=False).reset_index(drop=True)
    res['rerank_rank'] = np.arange(1, len(res) + 1)
    return res

# MODE 1: X answers per selected scholar (each scholar dataset searched independently)
def search_mode_per_scholar(query: str, scholars: list[str], x_per_scholar: int = 2, pool_per_scholar: int = 10) -> pd.DataFrame:
    q_vec = encode_query(query)
    results = []
    for scholar in scholars:
        scholar_mask = (df['source'] == scholar).values
        scholar_indices = np.where(scholar_mask)[0]
        if len(scholar_indices) == 0:
            continue
        sub_emb = embeddings[scholar_indices]
        sims = torch.mv(sub_emb, q_vec)
        k_cand = min(pool_per_scholar, len(scholar_indices))
        top_vals, top_rel_idx = torch.topk(sims, k=k_cand)
        top_abs_idx = scholar_indices[top_rel_idx.numpy()]
        cand_df = df.iloc[top_abs_idx].copy().reset_index(drop=True)
        cand_df['retrieval_rank'] = np.arange(1, len(cand_df) + 1)
        cand_df['embedding_score'] = top_vals.numpy()
        reranked = rerank(query, cand_df)
        results.append(reranked.head(x_per_scholar))
    if results:
        out = pd.concat(results, ignore_index=True)
        out['global_rank'] = np.arange(1, len(out) + 1)
        return out
    return pd.DataFrame()

# MODE 2: X answers ranked overall (across all selected scholars)
def search_mode_overall(query: str, scholars: list[str], x_total: int = 5, pool_total: int = 20) -> pd.DataFrame:
    q_vec = encode_query(query)
    scholar_mask = df['source'].isin(scholars).values
    indices = np.where(scholar_mask)[0]
    if len(indices) == 0:
        return pd.DataFrame()
    sub_emb = embeddings[indices]
    sims = torch.mv(sub_emb, q_vec)
    k_cand = min(pool_total, len(indices))
    top_vals, top_rel_idx = torch.topk(sims, k=k_cand)
    top_abs_idx = indices[top_rel_idx.numpy()]
    cand_df = df.iloc[top_abs_idx].copy().reset_index(drop=True)
    cand_df['retrieval_rank'] = np.arange(1, len(cand_df) + 1)
    cand_df['embedding_score'] = top_vals.numpy()
    reranked = rerank(query, cand_df)
    return reranked.head(x_total).reset_index(drop=True)

In [5]:
# 5. AI Synthesis: Consensus Classification & Comparative Summary

def call_gemini_resilient(client, model: str, contents: str, config: dict = None, max_retries: int = 3):
    """Calls Gemini with exponential backoff on transient 503/429 spikes."""
    delay = 2
    for attempt in range(max_retries):
        try:
            return client.models.generate_content(model=model, contents=contents, config=config)
        except Exception as e:
            err_str = str(e)
            if attempt < max_retries - 1 and ('503' in err_str or 'UNAVAILABLE' in err_str or '429' in err_str):
                print(f'Notice: Gemini temporary load spike ({type(e).__name__}). Retrying in {delay}s...')
                time.sleep(delay)
                delay *= 2
            else:
                raise

def format_fatwas_for_llm(candidates: pd.DataFrame, max_chars: int = 1500) -> str:
    lines = []
    for i, row in candidates.iterrows():
        lines.append(f"--- Fatwa [{i+1}] by {row['source']} ---")
        lines.append(f"Title: {row['title']}")
        lines.append(f"Question: {str(row['question'])[:300]}")
        lines.append(f"Answer: {str(row['answer'])[:max_chars]}\n")
    return '\n'.join(lines)

def classify_consensus_gemini(user_query: str, candidates: pd.DataFrame) -> dict:
    """Determines whether the retrieved scholars agree, partially agree, or disagree."""
    if not os.getenv('GEMINI_API_KEY'):
        return {'status': 'skipped', 'reason': 'GEMINI_API_KEY not set'}
    from google import genai
    client = genai.Client(api_key=os.getenv('GEMINI_API_KEY'))
    prompt = (
        f"User Question: {user_query}\n\n"
        f"Scholarly Fatwas:\n{format_fatwas_for_llm(candidates)}\n\n"
        "Task: Compare the rulings and positions of the scholars regarding the user question.\n"
        "Determine their level of agreement on the core ruling into one of three classes:\n"
        "- 'agree': All scholars share the same or harmonized ruling/stance.\n"
        "- 'partially-agree': General agreement on the fundamental ruling, with slight divergence in details, conditions, or exceptions.\n"
        "- 'disagree': Evident disagreement or differing rulings (e.g., permissible vs prohibited, obligatory vs recommended).\n\n"
        "Return a valid JSON object with keys: 'consensus' ('agree'|'partially-agree'|'disagree'), 'confidence' (float 0.0-1.0), and 'explanation' (Arabic explanation of consensus)."
    )
    resp = call_gemini_resilient(
        client,
        model='gemini-2.5-flash',
        contents=prompt,
        config={'response_mime_type': 'application/json'}
    )
    return json.loads(resp.text)

def summarize_positions_gemini(user_query: str, candidates: pd.DataFrame) -> str:
    """Produces an objective Arabic summary synthesizing the scholarly viewpoints."""
    if not os.getenv('GEMINI_API_KEY'):
        return 'Gemini API key not configured.'
    from google import genai
    client = genai.Client(api_key=os.getenv('GEMINI_API_KEY'))
    prompt = (
        f"السؤال: {user_query}\n\n"
        f"فتاوى العلماء:\n{format_fatwas_for_llm(candidates)}\n\n"
        "المطلوب: قدم ملخصاً فقهياً جامعاً وواضحاً باللغة العربية يوضح موقف العلماء وأقوالهم بشأن هذه المسألة، "
        "مع نسبة كل قول إلى الشيخ أو الهيئة التي أفتت به بإيجاز ودقة وبدون تحيز."
    )
    resp = call_gemini_resilient(
        client,
        model='gemini-2.5-flash',
        contents=prompt
    )
    return resp.text.strip()

In [6]:
# 6. Execution & Demo: Mode 1 (X answers per Scholar)
test_query = 'ما هي كفارة اليمين ومتى يشرع الإطعام ومتى يصام؟'
available_scholars = df['source'].unique().tolist()
print(f'Query: {test_query}\nAvailable scholars: {available_scholars}\n')

print('=== EXECUTING SEARCH MODE 1 (2 answers per scholar) ===')
results_mode1 = search_mode_per_scholar(test_query, available_scholars, x_per_scholar=2)
for i, r in results_mode1.iterrows():
    print(f"[{r['source']}] Fatwa #{r['fatwa_id']}: {r['title']} (Rerank score: {r['reranker_score']:.3f})")

Query: ما هي كفارة اليمين ومتى يشرع الإطعام ومتى يصام؟
Available scholars: ['Ibn Baz', 'Ibn Uthaymeen', 'Saleh Al-Fawzan', 'Al-Lajnah Ad-Daimah', 'Abdulaziz Aal ash-Sheikh', 'Abdullah al-Manee', 'Saleh bin Humaid']

=== EXECUTING SEARCH MODE 1 (2 answers per scholar) ===


[Ibn Baz] Fatwa #8913: حكم كفارة اليمين لتارك الصلاة (Rerank score: 0.972)
[Ibn Baz] Fatwa #16353: ما هي كفارة اليمين؟ (Rerank score: 0.972)
[Ibn Uthaymeen] Fatwa #27527: الأيمان والنذور (Rerank score: 0.971)
[Ibn Uthaymeen] Fatwa #27618: الأيمان والنذور (Rerank score: 0.969)
[Saleh Al-Fawzan] Fatwa #29203: كفارة اليمين. (Rerank score: 0.972)
[Saleh Al-Fawzan] Fatwa #29166: كفَّارة اليمين المُغلَّظة (Rerank score: 0.933)
[Al-Lajnah Ad-Daimah] Fatwa #30757: الإيلاء (Rerank score: 0.927)
[Al-Lajnah Ad-Daimah] Fatwa #34662: كفارة الصيام (Rerank score: 0.797)
[Abdulaziz Aal ash-Sheikh] Fatwa #39671: كيفية التكفير عن الأيمان والنذور (Rerank score: 0.657)
[Abdulaziz Aal ash-Sheikh] Fatwa #39693: فطر من ابتدأ الصيام في بلد وأفطر في بلد آخر (Rerank score: 0.317)
[Abdullah al-Manee] Fatwa #39870: قضاء الصلوات الفوائت (Rerank score: 0.051)
[Abdullah al-Manee] Fatwa #40009: حكم العادة السرية في رمضان (Rerank score: 0.039)
[Saleh bin Humaid] Fatwa #40015: حكم نذر المسلم عند الغضب ، وهل تجب عليه ال

In [7]:
# 7. Execution & Demo: Mode 2 (Top 5 Overall)
print('=== EXECUTING SEARCH MODE 2 (Top 5 Overall across scholars) ===')
results_mode2 = search_mode_overall(test_query, available_scholars, x_total=5)
for i, r in results_mode2.iterrows():
    print(f"#{i+1} [{r['source']}] {r['title']} (Rerank score: {r['reranker_score']:.3f})")

=== EXECUTING SEARCH MODE 2 (Top 5 Overall across scholars) ===


#1 [Saleh Al-Fawzan] كفارة اليمين. (Rerank score: 0.972)
#2 [Ibn Baz] حكم كفارة اليمين لتارك الصلاة (Rerank score: 0.972)
#3 [Ibn Baz] ما هي كفارة اليمين؟ (Rerank score: 0.972)
#4 [Ibn Baz] توضيح كيفية كفارة اليمين (Rerank score: 0.972)
#5 [Ibn Baz] حكم من حلف على عدم فعل شيء ثم فعله (Rerank score: 0.972)


In [8]:
# 8. Demo: AI Consensus & Summary
if gemini_ok and len(results_mode1) > 0:
    print('Classifying scholarly consensus with Gemini...')
    consensus = classify_consensus_gemini(test_query, results_mode1)
    print('Consensus Result:', json.dumps(consensus, ensure_ascii=False, indent=2))
    
    print('\nGenerating Scholarly Summary with Gemini...')
    summary = summarize_positions_gemini(test_query, results_mode1)
    print('\n--- Scholarly Summary ---')
    print(summary)
else:
    print('Set GEMINI_API_KEY in .env to test real-time AI consensus classification and summary.')

Classifying scholarly consensus with Gemini...


Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Notice: Gemini temporary load spike (ServerError). Retrying in 2s...


Notice: Gemini temporary load spike (ServerError). Retrying in 4s...


Consensus Result: {
  "consensus": "agree",
  "confidence": 1.0,
  "explanation": "أجمع العلماء على أن كفارة اليمين هي على التخيير أولاً بين ثلاثة أمور: إطعام عشرة مساكين، أو كسوتهم، أو تحرير رقبة مؤمنة. فإذا عجز الحانث عن هذه الثلاثة، ينتقل إلى الخيار الرابع وهو صيام ثلاثة أيام. وقد شدد العلماء على أن الصيام لا يجزئ إلا عند العجز عن الإطعام أو الكسوة أو العتق، ومن صام وهو قادر على أحد الأمور الثلاثة الأولى فصيامه لا يكفي وعليه إخراج الكفارة الصحيحة. أما عن تفاصيل الإطعام، فقد ذكروا أنه يكون بإطعام عشرة مساكين وجبة طعام مشبعة، أو تمليكهم مقداراً معيناً من قوت البلد كالأرز أو التمر (كمثل نصف صاع لكل مسكين أي ما يقارب كيلو ونصف)، وذكروا أن صيام الأيام الثلاثة الأفضل أن يكون متتابعاً."
}

Generating Scholarly Summary with Gemini...



--- Scholarly Summary ---
إليكم ملخص فقهي جامع وواضح حول كفارة اليمين، مستخلصاً من فتاوى العلماء المذكورين:

**تعريف كفارة اليمين وأساسها الشرعي:**
كفارة اليمين هي ما أوجبه الله تعالى على من حلف يميناً منعقدة ثم حنث فيها (أي خالفها). وقد بين الله سبحانه وتعالى هذه الكفارة في كتابه الكريم بسورة المائدة، الآية 89، حيث يقول: ﴿لا يُؤَاخِذُكُمُ اللَّهُ بِاللَّغْوِ فِي أَيْمَانِكُمْ وَلَكِنْ يُؤَاخِذُكُمْ بِمَا عَقَّدْتُمُ الأَيْمَانَ فَكَفَّارَتُهُ إِطْعَامُ عَشَرَةِ مَسَاكِينَ مِنْ أَوْسَطِ مَا تُطْعِمُونَ أَهْلِيكُمْ أَوْ كِسْوَتُهُمْ أَوْ تَحْرِيرُ رَقَبَةٍ فَمَنْ لَمْ يَجِدْ فَصِيَامُ ثَلاثَةِ أَيَّامٍ ذَلِكَ كَفَّارَةُ أَيْمَانِكُمْ إِذَا حَلَفْتُمْ وَاحْفَظُوا أَيْمَانَكُمْ﴾.

**مكونات كفارة اليمين (على التخيير أولاً ثم الترتيب):**

يجمع العلماء على أن كفارة اليمين تتضمن ثلاثة خيارات على التخيير، ثم خيار رابع على الترتيب إذا عجز المكفر عن الأول والثاني والثالث:

1.  **الإطعام:**
    *   **متى يشرع الإطعام:** يشرع الإطعام كأحد الخيارات الثلاثة الأولى التي يختار منها المكفر عند قدرته ع